## Step 1: Initialize the PageIndex Client
Connect to the PageIndex API using your API key. This sets up the client that will handle document processing and question answering without using traditional vector embeddings.

In [ ]:
from pageindex import PageIndexClient
pi_client = PageIndexClient(api_key="your api key")


## Step 2: Define Benchmark Questions & Similarity Function
Create the answer key with 5 ground-truth questions and their expected answers from the budget PDF. Also define the similarity function that will grade the AI's responses by comparing them character-by-character.

In [ ]:
from difflib import SequenceMatcher
import pandas as pd

# Ground truth benchmark questions
benchmark = [
    {
        "question": "What is the estimated nominal GDP growth rate for 2025-26?",
        "expected": "The nominal GDP growth is estimated at 10.1%."
    },
    {
        "question": "What is the new rebate limit under the revised new income tax regime?",
        "expected": "Annual income of up to Rs 12 lakh will now receive a 100% rebate on taxable income."
    },
    {
        "question": "Which ministry received the highest budget allocation in 2025-26, and what is the amount?",
        "expected": "The Ministry of Defence received the highest allocation with Rs 6,81,210 crore."
    },
    {
        "question": "What is the proposed fiscal deficit target as a percentage of GDP for 2025-26?",
        "expected": "The fiscal deficit is targeted at 4.4% of GDP."
    },
    {
        "question": "What is the total allocation for the Pradhan Mantri Awas Yojana (Rural + Urban) in the new budget?",
        "expected": "It has been allocated Rs 78,126 crore."
    }
]

def similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio() * 100

## Step 3: Submit Document for Processing
Upload the budget PDF to PageIndex for processing. The API returns a unique `doc_id` that we use to reference this document in future queries.

In [ ]:
result = pi_client.submit_document("./budget.pdf")
primary_doc_id = result["doc_id"]


## Step 4: Check Document Processing Status
Verify that PageIndex has finished processing the uploaded document before we start asking questions.

In [ ]:
status = pi_client.get_document(primary_doc_id)["status"]
if status == "completed":
    print('Document processing completed')
    

## Step 5: Ask a Sample Question
Test the system with a general question to confirm it can retrieve and answer queries from the processed document.

In [ ]:
response = pi_client.chat_completions(
    messages=[{"role": "user", "content": "What are the key findings in this document?"}],
    doc_id=primary_doc_id
)
 
print(response["choices"][0]["message"]["content"])

## Step 6: View Document Tree Structure
Retrieve and display the internal tree structure that PageIndex built from the document. This shows how PageIndex organizes the content without using vector embeddings.

In [ ]:
tree_result = pi_client.get_tree(primary_doc_id)["result"]
print(tree_result)

## Step 7: Run Benchmark Evaluation & Calculate Accuracy
Ask all 5 benchmark questions one by one, compare each AI response to the expected answer using the similarity function, and calculate the overall PageIndex RAG accuracy score.

In [ ]:
pageindex_results = []

# Question 1: What is the total budget allocation?
item1 = benchmark[0]
response1 = pi_client.chat_completions(
    messages=[{"role": "user", "content": item1["question"]}],
    doc_id=primary_doc_id
)
answer1 = response1["choices"][0]["message"]["content"]
score1 = similarity(answer1, item1["expected"])
pageindex_results.append({
    "Question": item1["question"],
    "Expected": item1["expected"],
    "Predicted": answer1,
    "Score": score1
})

# Question 2: What is the allocation for education?
item2 = benchmark[1]
response2 = pi_client.chat_completions(
    messages=[{"role": "user", "content": item2["question"]}],
    doc_id=primary_doc_id
)
answer2 = response2["choices"][0]["message"]["content"]
score2 = similarity(answer2, item2["expected"])
pageindex_results.append({
    "Question": item2["question"],
    "Expected": item2["expected"],
    "Predicted": answer2,
    "Score": score2
})

# Question 3: What is the allocation for healthcare?
item3 = benchmark[2]
response3 = pi_client.chat_completions(
    messages=[{"role": "user", "content": item3["question"]}],
    doc_id=primary_doc_id
)
answer3 = response3["choices"][0]["message"]["content"]
score3 = similarity(answer3, item3["expected"])
pageindex_results.append({
    "Question": item3["question"],
    "Expected": item3["expected"],
    "Predicted": answer3,
    "Score": score3
})

# Question 4: Which sector received the highest budget allocation?
item4 = benchmark[3]
response4 = pi_client.chat_completions(
    messages=[{"role": "user", "content": item4["question"]}],
    doc_id=primary_doc_id
)
answer4 = response4["choices"][0]["message"]["content"]
score4 = similarity(answer4, item4["expected"])
pageindex_results.append({
    "Question": item4["question"],
    "Expected": item4["expected"],
    "Predicted": answer4,
    "Score": score4
})

# Question 5: What are the key highlights of the budget?
item5 = benchmark[4]
response5 = pi_client.chat_completions(
    messages=[{"role": "user", "content": item5["question"]}],
    doc_id=primary_doc_id
)
answer5 = response5["choices"][0]["message"]["content"]
score5 = similarity(answer5, item5["expected"])
pageindex_results.append({
    "Question": item5["question"],
    "Expected": item5["expected"],
    "Predicted": answer5,
    "Score": score5
})

pageindex_df = pd.DataFrame(pageindex_results)

pageindex_accuracy = pageindex_df["Score"].mean()

print("PageIndex Accuracy:", round(pageindex_accuracy, 2), "%")
pageindex_df
